# RIME-X Regional Averages Pipeline
Step-by-step pipeline to compute area-weighted regional climate averages from CMIP6 NetCDF files using sub-national masks.

In [ ]:
# ── CELL 1: Imports ──────────────────────────────────────────
import xarray as xr
import numpy as np
import pandas as pd
import os
import gc
import time

print("All imports successful!")

In [ ]:
# ── CELL 2: Configuration — update these paths ───────────────

SIMULATION_DIR = './GCM_models/CNRM-ESM2-1/RegridedMonthly'
TARGET_DIR     = './GCM_models/CNRM-ESM2-1/regional_averages_output'
MASKS_PATH     = './masks/subregion_masks.nc'
INDICATOR      = 'tas'   # change to 'pr', 'tasmax' etc. if needed

SCENARIOS_I_WANT = [
    'historical',
    'ssp126',
    'ssp245',
    'ssp370',
    'ssp460',
    'ssp585',
]

print(f"Simulation dir : {SIMULATION_DIR}")
print(f"Target dir     : {TARGET_DIR}")
print(f"Masks path     : {MASKS_PATH}")
print(f"Indicator      : {INDICATOR}")
print(f"Mask exists    : {os.path.exists(MASKS_PATH)}")
print(f"Sim dir exists : {os.path.exists(SIMULATION_DIR)}")

In [ ]:
# ── CELL 3: Load masks and apply cos(lat) area weighting ─────

masks = xr.open_dataset(MASKS_PATH)

# Apply cosine latitude weighting (accounts for smaller grid cells near poles)
lat_da  = xr.DataArray(masks.lat.values, dims='lat')
cos_lat = np.cos(np.deg2rad(lat_da))
cos_lat = xr.DataArray(
    cos_lat, coords=[lat_da], dims=['lat']
).broadcast_like(masks)

masks           = masks * cos_lat
masks['m_GLOBAL'] = cos_lat   # add global mask

print(f"Masks loaded successfully!")
print(f"Total regions in mask : {len(list(masks.data_vars))}")
print(f"Sample region names   : {list(masks.data_vars)[:5]}")
print(f"Grid shape            : lat={len(masks.lat)}, lon={len(masks.lon)}")

In [ ]:
# ── CELL 4: Find matching NetCDF files ───────────────────────

all_files = [
    f for f in os.listdir(SIMULATION_DIR)
    if f.endswith('.nc')
    and len(f.split('_')) > 3
    and f.split('_')[3] in SCENARIOS_I_WANT
]

print(f"Files found: {len(all_files)}")
for f in all_files:
    parts = f.split('_')
    print(f"  {f}")
    print(f"    indicator={parts[0]}, model={parts[2]}, scenario={parts[3]}, ensemble={parts[4]}")

In [ ]:
# ── CELL 5: Define the regional averages function ────────────

def create_regional_averages_cmip6(
    simulation_directory,
    simulation_file,
    TARGET_DIRECTORY,
    masks,
    indicator,
    need_global_mean=False,
):
    simulation_path = f'{simulation_directory}/{simulation_file}'

    MODEL     = simulation_file.split('_')[2]
    scenario  = simulation_file.split('_')[3]
    ensemble  = simulation_file.split('_')[4]
    SCENARIO  = f'{scenario}-{ensemble}'
    INDICATOR = simulation_file.split('_')[0]

    print(f"\nProcessing: {MODEL} | {SCENARIO} | {INDICATOR}")
    print(f"  Loading: {simulation_path}")

    # Load NetCDF
    simulation = xr.open_dataset(simulation_path).load()

    # Fix lon convention: 0-360 → -180 to 180
    simulation['lon'] = xr.where(
        simulation['lon'] > 180,
        simulation['lon'] - 360,
        simulation['lon']
    )
    simulation = simulation.sortby('lon')

    # Align mask to simulation grid
    masks_aligned = masks.broadcast_like(simulation.isel(time=0))
    masks_aligned = xr.where(masks_aligned == 0, np.nan, masks_aligned)

    # Area-weighted regional average
    averages = masks_aligned * simulation[indicator]
    averages = averages.sum(dim=['lat', 'lon']) / masks_aligned.sum(dim=['lat', 'lon'])

    # Save global mean CSV (tas only)
    if need_global_mean and 'm_GLOBAL' in averages:
        df = pd.DataFrame({
            'time' : averages['time'].values,
            'tas'  : averages['m_GLOBAL'].values,
        })
        gm_dir = f'{TARGET_DIRECTORY}/cmip-6_global_mean/{indicator}'
        os.makedirs(gm_dir, exist_ok=True)
        gm_path = f'{gm_dir}/globalmean_{INDICATOR.lower()}_{MODEL.lower()}_{SCENARIO.lower()}.csv'
        df.to_csv(gm_path, index=False)
        print(f"  Global mean saved → {gm_path}")

    # Save one CSV per region
    region_count = 0
    for region in list(averages.keys()):
        region_clean = region.replace('m_', '')
        df = pd.DataFrame({
            'time'       : averages['time'].values,
            region_clean : averages[region].values,
        })
        directory = f'{TARGET_DIRECTORY}/isimip_regional_data/{region_clean}/latWeight'
        filename  = f'{MODEL}_{SCENARIO}_{INDICATOR}_{region_clean}_latweight.csv'.lower()
        os.makedirs(directory, exist_ok=True)
        df.to_csv(f'{directory}/{filename}', index=False)
        region_count += 1

    print(f"  Regional CSVs saved: {region_count} regions")

    # Cleanup memory
    simulation.close()
    del simulation, averages, masks_aligned
    gc.collect()
    time.sleep(2)

print("Function defined successfully!")

In [ ]:
# ── CELL 6: Run the pipeline ─────────────────────────────────

need_global_mean = (INDICATOR == 'tas')

print(f"Starting pipeline for {len(all_files)} file(s)...")
print(f"Global mean export: {need_global_mean}")
print("=" * 50)

for file in all_files:
    create_regional_averages_cmip6(
        SIMULATION_DIR,
        file,
        TARGET_DIR,
        masks,
        INDICATOR,
        need_global_mean,
    )

print("=" * 50)
print("PIPELINE COMPLETE!")

In [ ]:
# ── CELL 7: Verify outputs ───────────────────────────────────

print("=== Global Mean CSVs ===")
gm_dir = f'{TARGET_DIR}/cmip-6_global_mean/{INDICATOR}'
if os.path.exists(gm_dir):
    for f in os.listdir(gm_dir):
        fpath = f'{gm_dir}/{f}'
        df = pd.read_csv(fpath)
        print(f"  {f}")
        print(f"    rows={len(df)}, years={df['time'].iloc[0]} to {df['time'].iloc[-1]}")
        print(f"    preview: {df.head(2).to_string(index=False)}")
else:
    print("  No global mean files found yet.")

print("\n=== Regional CSV folders ===")
reg_dir = f'{TARGET_DIR}/isimip_regional_data'
if os.path.exists(reg_dir):
    regions = os.listdir(reg_dir)
    print(f"  Total regions with output: {len(regions)}")
    print(f"  Sample regions: {regions[:5]}")

    # Preview one region CSV
    if regions:
        sample_region = regions[0]
        lw_dir = f'{reg_dir}/{sample_region}/latWeight'
        if os.path.exists(lw_dir):
            sample_file = os.listdir(lw_dir)[0]
            df = pd.read_csv(f'{lw_dir}/{sample_file}')
            print(f"\n  Sample file: {sample_file}")
            print(f"  {df.head(3).to_string(index=False)}")
else:
    print("  No regional output found yet.")